# week 1 - streaming structured output

In [1]:
import os
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    max_retries=10,
)

MODEL = "openai/gpt-oss-20b"

## streaming recap

gpt-oss is a reasoning model, the stream also has `response.reasoning_text.delta` events, so filter on `event.type` instead of `hasattr(event, 'delta')`

In [2]:
messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

stream = openai_client.responses.create(
    model=MODEL,
    input=messages,
    stream=True
)

response = None
for event in stream:
    if event.type == 'response.output_text.delta':
        print(event.delta, end='')
    if hasattr(event, 'response'):
        response = event.response

**The Day the Rainbow Got Stuck**

Once upon a time in the misty valleys of Glimmerglen, there lived a young unicorn named Liora. She was bright, hopeful, and had a shimmering silver horn that glittered whenever she trotted through the dew‑kissed meadows. Liora had a secret dream: every sunrise, she wished to paint the sky with a rainbow that would last a whole day.

On a bright, hopeful morning, Liora pranced to the ancient crystal hill where the rainbow was said to be born. She closed her eyes, inhaled the scent of wild jasmine, and let her heart beat in perfect rhythm. As she released a gentle sigh, a swirl of colors burst forth from her horn, stretching across the horizon. It was a magnificent rainbow, more dazzling than any before.

But as the sun climbed higher, the colors began to fray. Liora’s horn felt heavy, as if it were carrying the weight of the world. The rainbow flickered, then snapped into a jagged line that hung uselessly over the valley. A sharp wind blew, and the rai

## structured output

In [3]:
from pydantic import BaseModel, Field

class ArticleSection(BaseModel):
    header: str
    text: str = Field(description="text of the section in markdown")

class ArticleResponse(BaseModel):
    title: str
    subtitle: str
    sections: list[ArticleSection]

In [4]:
try:
    response = openai_client.responses.parse(
        model=MODEL,
        input=messages,
        text_format=ArticleResponse,
        stream=True  # this doesn't work
    )
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

Error: AttributeError: 'str' object has no attribute 'output'


In [5]:
with openai_client.responses.stream(
    model=MODEL,
    input=messages,
    text_format=ArticleResponse,
) as stream:
    response = None
    n_deltas = 0
    for event in stream:
        if event.type == 'response.output_text.delta':
            n_deltas += 1
            print(event.delta, end='')
        if hasattr(event, 'response'):
            response = event.response

print()
print('deltas:', n_deltas)

{"title":"The Midnight Misfortune of the Moonlit Unicorn","subtitle":"A tale of bad times and lost hope","sections":[{"header":"The Dawn of Despair","text":"In a valley where the sky never rose, a unicorn named Liora once shone like a fallen star. She prided herself on the brilliance of her mane, the purity of her horn, and the hope she spread to the wandering souls. But as the world turned, darkness crept in, and her light began to flicker."},{"header":"The Curse Unleashed","text":"One fateful night, a jealous witch named Mirelle whispered a curse over the moonlit glade. She cursed the unicorn's glow to fade with every sorrow, binding Liora's destiny to the shadows of every heart she touched."},{"header":"Lost Spark","text":"Liora felt the first tremor in her horn. A dullness spread through her mane, and the world around her grew bleak. Every joy she offered seemed to pull her further into sorrow, as if the curse fed on the kindness she had once shared."},{"header":"A Glimmer of Redem

groq sends the structured json in one single delta, so nothing actually streams here. more on that below

In [6]:
story = response.output_parsed

print('# ' + story.title)
print(story.subtitle)
print()

for section in story.sections:
    print('## ' + section.header)
    print()
    print(section.text)
    print()
    print()

# The Midnight Misfortune of the Moonlit Unicorn
A tale of bad times and lost hope

## The Dawn of Despair

In a valley where the sky never rose, a unicorn named Liora once shone like a fallen star. She prided herself on the brilliance of her mane, the purity of her horn, and the hope she spread to the wandering souls. But as the world turned, darkness crept in, and her light began to flicker.


## The Curse Unleashed

One fateful night, a jealous witch named Mirelle whispered a curse over the moonlit glade. She cursed the unicorn's glow to fade with every sorrow, binding Liora's destiny to the shadows of every heart she touched.


## Lost Spark

Liora felt the first tremor in her horn. A dullness spread through her mane, and the world around her grew bleak. Every joy she offered seemed to pull her further into sorrow, as if the curse fed on the kindness she had once shared.


## A Glimmer of Redemption

Yet in her darkest hour, a small child with a chipped tooth and a hopeful grin whi

## parsing json incrementally with jaxn

In [7]:
from typing import Any, Dict

from jaxn import JSONParserHandler, StreamingJSONParser

In [8]:
raw_json = """
{"title":"The Misadventures of Sparkle","subtitle":"A Tale of Trouble","sections":[{"header":"Once Upon a Time","text":"In a magical land far away, there lived a young unicorn named Sparkle."},{"header":"The Problem","text":"Sparkle had a mischievous streak that often led to chaos."}]}
""".strip()

In [9]:
class ArticleResponseHandler(JSONParserHandler):

    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '':
            if field_name == 'title':
                print(f'# {value}')
            if field_name == 'subtitle':
                print(value)
        if path == '/sections' and field_name == 'header':
            print(f'\n\n## {value}\n')

    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '/sections' and field_name == 'text':
            print(chunk, end='', flush=True)

In [10]:
handler = ArticleResponseHandler()
parser = StreamingJSONParser(handler=handler)

for c in raw_json:
    parser.parse_incremental(c)

# The Misadventures of Sparkle
A Tale of Trouble


## Once Upon a Time

In a magical land far away, there lived a young unicorn named Sparkle.

## The Problem

Sparkle had a mischievous streak that often led to chaos.

## llm_structured_stream

same as the lesson, but only `output_text` deltas go to the parser (otherwise the reasoning text ends up in there too)

In [11]:
def llm_structured_stream(
    user_prompt,
    output_type,
    parser_handler=JSONParserHandler(),
    instructions=None,
    model=MODEL,
):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    parser = StreamingJSONParser(handler=parser_handler)

    with openai_client.responses.stream(
        model=model,
        input=messages,
        text_format=output_type,
    ) as stream:
        response = None
        for event in stream:
            if event.type == 'response.output_text.delta':
                parser.parse_incremental(event.delta)
            if hasattr(event, 'response'):
                response = event.response

    return response

In [12]:
instructions = "your task is to tell the user bad time stories"
user_prompt = "unicorn"

result = llm_structured_stream(
    instructions=instructions,
    user_prompt=user_prompt,
    output_type=ArticleResponse,
    parser_handler=ArticleResponseHandler(),
)

# The Time-Traveling Unicorn
A Collection of Bad Time Tales


## The Dawn of the Curse

In the first millennia, a unicorn named *Celestia* discovered a shimmering clock buried in the forest. When she touched it, the world spun backward, turning spring into winter.

*Bad Time 1:* The unicorn could no longer feel joy, her hooves sinking into the earth as time reversed, erasing the memory of laughter.


## The Forgotten Year

Centuries later, a weary traveler named *Liam* stumbled upon the clock again. He set it to 1999, hoping for nostalgia. Instead, the universe collapsed into a perpetual 1977.

*Bad Time 2:* 1977's disco craze clung to modern life, forcing everyone to dance until exhaustion, never reaching the future they sought.


## The Lost Hour

In the present day, a young coder named *Aria* tried to fix the clock. She misaligned the gears, and an entire hour vanished, leaving a void where daylight should have been.

*Bad Time 3:* The sun never rose that hour, and the city slept in

works, but because of groq everything shows up at once at the end.

workaround: groq streams normal text fine, so don't pass `text_format`. put the json schema in the instructions instead, stream the text through jaxn, and validate with pydantic at the end.
downside: the schema isn't enforced while generating, so the final `model_validate_json` is what catches bad output

In [13]:
import json

def llm_json_stream(
    user_prompt,
    output_type,
    parser_handler=JSONParserHandler(),
    instructions=None,
    model=MODEL,
):
    schema = json.dumps(output_type.model_json_schema())
    system = (instructions or '') + (
        "\n\nRespond only with JSON matching this schema, no markdown fences:\n" + schema
    )

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_prompt},
    ]

    parser = StreamingJSONParser(handler=parser_handler)
    text = ''

    with openai_client.responses.stream(
        model=model,
        input=messages,
    ) as stream:
        for event in stream:
            if event.type == 'response.output_text.delta':
                text += event.delta
                parser.parse_incremental(event.delta)

    return output_type.model_validate_json(text)

In [14]:
story = llm_json_stream(
    instructions="your task is to tell the user bad time stories",
    user_prompt="unicorn",
    output_type=ArticleResponse,
    parser_handler=ArticleResponseHandler(),
)

# The Tragic Tale of the Lost Unicorn
A Story of Misfortune and Resilience


## The Awakening

In a **silenced forest** where the wind once sang, a lone unicorn awoke to a world devoid of color. The sky was gray, the grass a dull shade of green, and every creature seemed to have lost its *spark*.

## The Descent

The unicorn struggled to find its way, only to discover that **magic was fading**. Its mane no longer shimmered, and each step it took left a trail of dust instead of stardust. The deeper it ventured into the abandoned glade, the more it felt the weight of a world that had forgotten *wonder*.

## A Glimmer of Hope

Yet, within the darkness, a tiny glimmer of light flickered—an *old oak tree* that still whispered stories of the past. The unicorn, with its heart still beating, remembered that even in the darkest times, hope could be found in the smallest of gestures. It decided to keep moving forward, determined to restore the magic that once filled the skies.

In [15]:
story.title, len(story.sections)

('The Tragic Tale of the Lost Unicorn', 3)

## streaming docs rag

In [16]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

Indexed 382 chunks from 95 documents


In [17]:
from typing import Literal

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [18]:
class RAGResponseHandler(JSONParserHandler):
    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '' and field_name == 'answer':
            print(chunk, end='', flush=True)

    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '' and field_name == 'answer_type':
            print('\nanswer type:', value)

    def on_array_item_end(self, path: str, field_name: str, item: Dict[str, Any] = None) -> None:
        if field_name == 'followup_questions':
            print('follow up question:', item)

using `llm_json_stream` here so the answer really streams

In [19]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_json_stream(
        instructions=instructions,
        user_prompt=prompt,
        output_type=RAGResponse,
        parser_handler=RAGResponseHandler()
    )

In [20]:
response = rag('llm as a judge')

## LLM as a Judge
An **LLM judge** is a lightweight evaluation system that uses a large language model to automatically score or classify the quality of other LLM outputs. It can be set up in two main modes:

1. **Reference‑based** – The judge compares a new response with a *ground‑truth* or reference response. It decides if the new answer is correct or not. This is useful for regression testing and for situations where you already have approved answers.
2. **Open‑ended** – The judge evaluates a response against custom criteria (e.g., verbosity, factuality, style) without needing a reference. The LLM is prompted to classify the text into one of two or more classes and can optionally provide reasoning.

### Key Steps from the Tutorial
1. **Create a toy dataset** containing *questions*, *target responses*, *new responses*, and *manual labels*.
2. **Define an LLM evaluator prompt** (e.g., `BinaryClassificationPromptTemplate`).
3. **Add the evaluator as a descriptor** to the dataset: `LLME

In [21]:
response.found_answer, response.confidence

(True, 0.97)